# Notebook for extracting mandatory subjects for all schools

In [ ]:
from firecrawl_app import app
from pydantic import BaseModel
import os
import sys
import json
import pandas as pd

# Optional if you want to import other helpers
sys.path.append(os.path.abspath(".."))

# Define schema
class DummySchema(BaseModel):
    obligatorise_emner: list[str]

# Try one prompt
def try_single_prompt(url, prompt):
    try:
        data = app.extract([url], {
            'prompt': prompt,
            'schema': DummySchema.model_json_schema()
        })
        return data.get("data", {}).get("obligatorise_emner", [])
    except Exception as e:
        print(f"Prompt failed: {prompt[:50]}...\nError: {e}")
        return []

# Load config
with open("config_mandatory.json", "r", encoding="utf-8") as f:
    config_data = json.load(f)

rows = []

# Loop through each school
for school, school_data in config_data.items():
    print(f"\n🏫 Now extracting mandatory subjects from {school}")

    for study_program, url in school_data["mandatory_subjects"].items():
        print(f" - Program: {study_program}")

        try:
            prompts = school_data["prompt_mandatory_subjects"]
            if isinstance(prompts, str):
                prompts = [prompts]

            best_result = []
            best_strategy = ""

            # Try each prompt separately
            for prompt in prompts:
                subjects = try_single_prompt(url, prompt)
                print(f"   ➤ Prompt: {prompt[:40]}... → {len(subjects)} subjects")
                if len(subjects) > len(best_result):
                    best_result = subjects
                    best_strategy = f"Single prompt: {prompt[:40]}..."

            # Try all prompts combined
            super_prompt = "\n".join(prompts)
            super_subjects = try_single_prompt(url, super_prompt)
            print(f"   🔁 Super prompt → {len(super_subjects)} subjects")

            # Use the best
            if len(super_subjects) > len(best_result):
                best_result = super_subjects
                best_strategy = "Super prompt"

            if best_result:
                for subject in best_result:
                    rows.append({
                        "Skole": school,
                        "Studie program": study_program,
                        "Læringsutbytte type": "Obligatorise_emner",
                        "Læringsutbytte": subject.strip(),
                    })
            else:
                print("   ❌ No mandatory subjects found.")
                rows.append({
                    "Skole": school,
                    "Studie program": study_program,
                    "Læringsutbytte type": "Obligatorise_emner",
                    "Læringsutbytte": "",
                })

        except Exception as e:
            print(f"   ❗ Failed to extract for {study_program}: {e}")
            rows.append({
                "Skole": school,
                "Studie program": study_program,
                "Læringsutbytte type": "Obligatorise_emner",
                "Læringsutbytte": "",
            })

# Save to CSV
df = pd.DataFrame(rows)
df.to_csv("Mandatory_subjects.csv", index=False)
print("\n✅ Done! Saved to Mandatory_subjects.csv")



🏫 Now extracting mandatory subjects from UIO
 - Program: Informatikk: Programmering og systemarkitektur
   ➤ Prompt: Extract all course names (emner) listed ... → 11 subjects
Prompt failed: Extract all course names (emner) listed as mandato...
Error: ("HTTPSConnectionPool(host='api.firecrawl.dev', port=443): Max retries exceeded with url: /v1/extract/a1fa5622-1c2c-4db0-942b-e79f952f651f (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1000)')))", 500)
   ➤ Prompt: Extract all course names (emner) listed ... → 0 subjects
   🔁 Super prompt → 11 subjects
 - Program: Informatikk: Design, bruk og interaksjon
   ➤ Prompt: Extract all course names (emner) listed ... → 12 subjects
   ➤ Prompt: Extract all course names (emner) listed ... → 13 subjects
   🔁 Super prompt → 11 subjects
 - Program: Informatikk: Digital oekonomi og ledelse
   ➤ Prompt: Extract all course names (emner) listed ... → 14 subjects
   ➤ Prompt: Extract 


🏫 Now extracting mandatory subjects from UIO
 - Program: Informatikk: Programmering og systemarkitektur
   ➤ Prompt: Extract all course names (emner) listed ... → 11 subjects
   🔁 Super prompt → 13 subjects
 - Program: Informatikk: Design, bruk og interaksjon
   ➤ Prompt: Extract all course names (emner) listed ... → 13 subjects
   🔁 Super prompt → 12 subjects
 - Program: Informatikk: Digital oekonomi og ledelse
   ➤ Prompt: Extract all course names (emner) listed ... → 15 subjects
   🔁 Super prompt → 14 subjects

🏫 Now extracting mandatory subjects from UIB
 - Program: Bachelor i informatikk, datateknologi
   ➤ Prompt: Extract all course names (emner) listed ... → 15 subjects
   🔁 Super prompt → 14 subjects
 - Program: Bachelor i kunstig intelligens
   ➤ Prompt: Extract all course names (emner) listed ... → 13 subjects
   🔁 Super prompt → 13 subjects

🏫 Now extracting mandatory subjects from UIA
 - Program: Bachelor i ingenioerfag, data
   ➤ Prompt: Find all elements marked as 'oblig

In [ ]:
""" import os
import sys
from firecrawl_app import app, ExtractSchema, NestedModel
import json
import pandas as pd
sys.path.append(os.path.abspath(".."))

from utils.helpers import create_csv
from utils.helpers import *

filepath = "config_mandatory.json"

# Load the JSON file containing config
with open(filepath, "r", encoding="utf-8") as json_file:
    config_data = json.load(json_file)
    
# Get all schools
schools = list(config_data.keys())

print("Schools currently defined in config_mandatory.json:")
print("-" * 40)
for school in sorted(schools):
    print(f"- {school}")  """

Schools currently defined in config_mandatory.json:
----------------------------------------
- HIOF
- KRISTIANIA
- NTNU
- UIA
- UIB
- UIO
- UIS
- UIT
- USN


## Code for running extraction of mandatory subjects for all schools defined in config_mandatory.json.

In [ ]:
""" first_write = True
output_file = "Second_Mandatory_subjects.csv"
rows = []

for school, school_data in config_data.items():
    print(f"\nNow extracting mandatory subjects from {school}")

    for study_program, url in school_data["mandatory_subjects"].items():
        print(f" - Extracting for program: {study_program}")
        
        try:
            data = app.extract([url], {
                'prompt': school_data["prompt_mandatory_subjects"],
                'schema': ExtractSchema.model_json_schema()
            })

            subjects = data["data"]["læringsutbyttebeskrivelser"].get("obligatorise_emner", [])

            if subjects:
                for subject in subjects:
                    rows.append({
                        "school": school,
                        "study_program": study_program,
                        "type": "Obligatorise_emner",
                        "mandatory_subject": subject.strip()
                    })
            else:
                print("    No mandatory subjects found.")
                rows.append({
                    "school": school,
                    "study_program": study_program,
                    "type": "Obligatorise_emner",
                    "mandatory_subject": ""
                })

        except Exception as e:
            print(f"   Failed to extract for {study_program}: {e}")
            # Log failure as an empty row too
            rows.append({
                "school": school,
                "study_program": study_program,
                "type": "Obligatorise_emner",
                "mandatory_subject": ""
            })

# Save all collected rows
df = pd.DataFrame(rows)
df.to_csv(output_file, index=False)
print("\n✅ Mandatory subjects written to CSV!")
 """


Now extracting mandatory subjects from UIO
 - Extracting for program: Informatikk: Programmering og systemarkitektur
 - Extracting for program: Informatikk: Design, bruk og interaksjon
 - Extracting for program: Informatikk: Digital oekonomi og ledelse

Now extracting mandatory subjects from UIB
 - Extracting for program: Bachelor i informatikk, datateknologi
 - Extracting for program: Bachelor i kunstig intelligens

Now extracting mandatory subjects from UIA
 - Extracting for program: Bachelor i ingenioerfag, data
   Failed to extract for Bachelor i ingenioerfag, data: ("Unexpected error during extract: Status code 400. Bad Request - [{'code': 'invalid_type', 'expected': 'string', 'received': 'array', 'path': ['prompt'], 'message': 'Expected string, received array'}]", 500)

Now extracting mandatory subjects from NTNU
 - Extracting for program: Bachelor i programmering
   Failed to extract for Bachelor i programmering: ("Unexpected error during extract: Status code 400. Bad Request -